# NeuroOmics-AD — End-to-end demo notebook

This notebook walks through the full platform workflow **via the REST API**: login → project → upload → DE → enrichment → network → ML → drugs → report → assistant.

**Prereqs:** backend running at `http://localhost:8000` (or `docker compose up -d`).

In [ ]:
import os, json, requests

BASE = os.environ.get("NEUROOMICS_API", "http://localhost:8000/api/v1")
s = requests.Session()

# login as demo user (seed with `python backend/scripts/seed_demo.py`)
r = s.post(f"{BASE}/auth/login", json={"email": "demo@neuroomics.org", "password": "demo12345"})
s.headers["Authorization"] = f"Bearer {r.json()['access_token']}"
print("login:", r.status_code)

In [ ]:
# list projects & datasets
projects = s.get(f"{BASE}/projects").json()
pid = projects[0]["id"]
datasets = s.get(f"{BASE}/datasets?project_id={pid}").json()
print("project:", projects[0]["name"])
for d in datasets:
    print("  dataset:", d["name"], "(", d["omics_type"], d["n_samples"], "samples )")
rna_id = next(d["id"] for d in datasets if d["omics_type"] == "transcriptomics")

In [ ]:
# 1) Differential expression (Celery-backed analysis)
r = s.post(f"{BASE}/analyses/{pid}/create", json={
    "name": "DE AD vs CN", "analysis_type": "differential_expression",
    "config": {"dataset_id": rna_id, "case_group": "AD", "control_group": "CN"}})
aid = r.json()["id"]
print("analysis:", r.status_code, r.json()["status"])
result = s.get(f"{BASE}/analyses/{aid}/result").json()
print("DE summary:", result["summary"])
top = [g["gene"] for g in result["table"] if g["sig"]][:10]
print("top genes:", top)

In [ ]:
# 2) Enrichment & network
enr = s.post(f"{BASE}/omics/enrichment", json={"gene_list": top}).json()
print("enrichment:", [e["pathway"] for e in enr["table"][:4]])
net = s.post(f"{BASE}/omics/network", json={"gene_list": top}).json()
print("hubs:", net["hub_genes"][:6])

In [ ]:
# 3) Machine learning (RF + GNN)
ml = s.post(f"{BASE}/ml/train", json={
    "dataset_id": rna_id, "label_column": "group",
    "algorithms": ["random_forest", "xgboost", "svm", "dnn", "gnn"],
    "cv_folds": 3, "top_features": 100}).json()
for m in ml["results"]:
    print(f"{m['algorithm']:14s} AUC={m['metrics'].get('roc_auc', 0):.3f}")

In [ ]:
# 4) Drug repurposing
drugs = s.post(f"{BASE}/drugs/pipeline", json={"gene_list": top, "max_candidates": 8}).json()
for c in drugs["candidates"][:5]:
    print(f"#{c['rank']} {c['drug_name']:22s} score={c['composite_score']:.3f} {c['mechanism'][:50]}")
print("combos:", [(x["drug_a"], x["drug_b"]) for x in drugs["combinations"][:3]])

In [ ]:
# 5) Report (PDF + Word + HTML)
rep = s.post(f"{BASE}/reports/generate", json={
    "analysis_ids": [aid], "formats": ["pdf", "docx", "html"], "dpi": 300}).json()
print("report files:", {k: v.split("/")[-1] for k, v in rep["files"].items()})

In [ ]:
# 6) AI assistant + manuscript
chat = s.post(f"{BASE}/assistant/chat", json={
    "message": "Which genes are most differentially expressed and what do they mean?",
    "project_id": pid, "analysis_ids": [aid]}).json()
print("assistant mode:", chat["mode"])
print(chat["reply"][:400])

ms = s.post(f"{BASE}/assistant/manuscript", json={"analysis_ids": [aid]}).json()
print("\n--- Results (first 300 chars) ---\n", ms["results"][:300])